# 1. PDF : **Resume Parsing**

In [1]:
# pip install pymupdf pandas

In [1]:
import dspy
import litellm
import os
import fitz
import httpx
import base64
import re

from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Optional, List  

# ปิดการแจ้งเตือน SSL
litellm.ssl_verify = False

# ==========================================
# 1. Configuration & Setup

LLM_URL = 'https://llm.services.storemesh.com/v1'
API_KEY = 'sk-qwwRBsI8WJfS1Yh-hBgyiQ'
DSPY_MODEL_NAME = 'gpt-oss:20b' 
OCR_MODEL_NAME = 'typhoon-ocr1.5-3b' 

# ตั้งค่า DSPy LM
lm = dspy.LM(
    model=f'openai/{DSPY_MODEL_NAME}',
    api_base=LLM_URL, 
    api_key=API_KEY, 
    cache=False
)
dspy.configure(lm=lm, track_usage=True)

# ==========================================
# 2. Exceptions & Validators

class PDFValidationError(Exception): pass
class LowTextDensityError(PDFValidationError): pass
class BrokenFontError(PDFValidationError): pass
class BrokenThaiTextError(PDFValidationError): pass
class HighImageCoverageError(PDFValidationError): pass
class ComplexVectorTableError(PDFValidationError): pass

class PDFValidator:
    @staticmethod
    def check_image_coverage(page: fitz.Page):
        page_area = page.rect.get_area()
        total_image_area = sum((img.get("bbox")[2] - img.get("bbox")[0]) * (img.get("bbox")[3] - img.get("bbox")[1]) 
                               for img in page.get_image_info() if img.get("bbox"))
        image_coverage = (total_image_area / page_area) if page_area > 0 else 0
        if image_coverage > 0.60:
            raise HighImageCoverageError(f"High image coverage: {image_coverage:.2%}")

    @staticmethod
    def check_complex_tables(page: fitz.Page):
        if len(page.get_drawings()) > 100: 
            raise ComplexVectorTableError(f"Complex vector tables detected")

    @staticmethod
    def check_text_density(text: str):
        if not text or len(text.strip()) < 30: 
            raise LowTextDensityError(f"Low text density")

    @staticmethod
    def check_broken_font(text: str):
        pua_count = len(re.findall(r'[\uf700-\uf7ff]', text))
        ufffd_count = text.count("\ufffd")
        mojibake_count = len(re.findall(r'[\u00C0-\u024F]', text))
        control_char_count = len(re.findall(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', text))
        math_error = "Math input error" in text
        
        if "(cid:" in text or ufffd_count > 1 or pua_count > 3 or mojibake_count > 2 or control_char_count > 5 or math_error:
            raise BrokenFontError(f"Broken Encoding: PUA={pua_count}, UFFFD={ufffd_count}, Mojibake={mojibake_count}, CtrlChars={control_char_count}")

    @staticmethod
    def check_broken_thai(text: str):
        broken_thai_pattern = r'(?:\s[\u0e30-\u0e3a\u0e45\u0e47-\u0e4e])|(?:[\u0e32\u0e33\u0e40-\u0e44][\u0e31\u0e34-\u0e3a\u0e47-\u0e4e])'
        broken_count = len(re.findall(broken_thai_pattern, text))
        if broken_count >= 1:
            raise BrokenThaiTextError(f"Broken Thai vowels/tones: {broken_count} occurrences")

    @classmethod
    def validate_layout(cls, page: fitz.Page):
        cls.check_image_coverage(page)
        cls.check_complex_tables(page)

    @classmethod
    def validate_text(cls, text: str):
        if not isinstance(text, str):
            text = str(text) if text is not None else ""
        cls.check_text_density(text)
        cls.check_broken_font(text)
        cls.check_broken_thai(text)


# ==========================================
# 3. OCR Engine (Typhoon) - *แก้ไขแล้ว*

def typhoon_parser(base64_image: str, prompt="Extract all text from the image into clean Markdown Format. Please keep original informations.", model_name=OCR_MODEL_NAME) -> str: 
    print(f"    [Typhoon OCR] 👁️ กำลังอ่านรูปภาพด้วย AI...")
    
    custom_http_client = httpx.Client(verify=False)
    client = OpenAI(
        base_url=LLM_URL, 
        api_key=API_KEY,
        http_client=custom_http_client
    )
    
    try:
        response = client.chat.completions.create(
            model=model_name,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            temperature=0.2,
            max_tokens=8192, 
            extra_body={
                "options": {
                    "num_ctx": 8192,  
                    "num_gpu": 99      
                }
            }
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"    ❌ Typhoon OCR Error: {e}")
        return ""


# ==========================================
# 4. Smart PDF Extractor - *แก้ไขแล้ว*

def smart_pdf_extractor(pdf_path: str) -> str:
    full_text = ""
    doc = fitz.open(pdf_path)
    
    for page_num in range(doc.page_count):
        page = doc.load_page(page_num)
        text = page.get_text("text")
        
        try:
            PDFValidator.validate_layout(page)
            PDFValidator.validate_text(text)
            
            full_text += text + "\n\n"
            print(f"📄 Page {page_num + 1}: อ่านด้วย Fitz ปกติสำเร็จ")
            
        except PDFValidationError as e:
            print(f"⚠️ Page {page_num + 1}: พบปัญหา '{e}' -> แปลงเป็น Base64 แล้วส่งให้ OCR...")
            
            # ดึงภาพจาก PDF แล้วแปลงเป็น Bytes เลย โดยไม่เซฟลงไฟล์
            pix = page.get_pixmap(matrix=fitz.Matrix(3, 3), dpi=150) 
            img_data = pix.tobytes("png")
            base64_image = base64.b64encode(img_data).decode('utf-8')
            
            # ส่ง Base64 เข้า OCR โดยตรง
            ocr_text = typhoon_parser(base64_image)
            full_text += ocr_text + "\n\n"
            
    doc.close()
    return full_text


# ==========================================
# 5. Pydantic Schema

class ExactProfileSchema(BaseModel):
    full_name: str
    student_id: Optional[str] = Field(None, description="Student ID")
    email: Optional[str]
    phone: Optional[str]
    location: Optional[str] = Field(None, description="City, Country in English")
    gpa: Optional[float] = Field(None, description="GPA as float")
    
    skill: List[str] = Field(
        description="List of professional and technical skills. "
                    "IMPORTANT: You MUST break down grouped skills into single, specific individual items. "
                    "For example, instead of 'Basic Programming (Java, Python, PHP)', "
                    "you must output ['Java', 'Python', 'PHP']. Do not include parentheses."
    )
    
    interests: List[str] = Field(description="Explicit Hobbies or Personal Interests only...")
    experience_summary: List[str] = Field(description="List of professional experiences...")
    trainings: List[str] = Field(description="List of trainings/workshops.")


# ==========================================
# 6. Post-Processing Function

def clean_extracted_skills(raw_skills: List[str]) -> List[str]:
    cleaned = []
    if not raw_skills:
        return cleaned
        
    for s in raw_skills:
        match = re.search(r'\((.*?)\)', s)
        if match:
            inner_skills = [x.strip() for x in match.group(1).split(',')]
            cleaned.extend(inner_skills)
        else:
            cleaned.extend([x.strip() for x in s.split(',')])
            
    return list(set(filter(None, cleaned)))


# ==========================================
# 7. DSPy Agent สำหรับจัดโครงสร้างด้วย Pydantic

class ResumeParsingSignature(dspy.Signature):
    """คุณคือ HR Assistant หน้าที่ของคุณคือสกัดข้อมูลจากเรซูเม่ (Raw Text) และจัดให้อยู่ในรูปแบบโครงสร้าง"""
    resume_raw_text: str = dspy.InputField(desc="ข้อความดิบที่สกัดมาจาก PDF หรือ OCR")
    parsed_profile: ExactProfileSchema = dspy.OutputField(desc="ข้อมูลโปรไฟล์ที่ถูกจัดโครงสร้างตาม Schema ที่กำหนด")

class ResumeParsingAgent(dspy.Module):
    def __init__(self):
        super().__init__()
        self.extractor = dspy.ChainOfThought(ResumeParsingSignature)
        
    def forward(self, pdf_path: str):
        print(f"\n🚀 เริ่มกระบวนการสกัดข้อความจาก: {pdf_path}")
        raw_text = smart_pdf_extractor(pdf_path)
        if not raw_text.strip():
            return None
        print(f"\n🧠 ส่งข้อความ (ความยาว {len(raw_text)} ตัวอักษร) ให้ DSPy Agent วิเคราะห์เป็น JSON...")
        result = self.extractor(resume_raw_text=raw_text)
        return result.parsed_profile

if __name__ == "__main__":
    # TARGET_PDF = './input/Resume_Bantoon - บัณฑูรย์ สุขแสงแก้ว.pdf'
    # TARGET_PDF = "./input/_Resume Sirichat - sirichat thueman.pdf"
    TARGET_PDF = "./input/kitichai - kitichai paichayon.pdf"
    if not os.path.exists(TARGET_PDF):
        print(f"❌ ไม่พบไฟล์: {TARGET_PDF}")
    else:
        agent = ResumeParsingAgent()
        profile_data = agent(pdf_path=TARGET_PDF)
        if profile_data:
            profile_data.skill = clean_extracted_skills(profile_data.skill)
            print("\n" + "="*50)
            print("🎯 ผลลัพธ์ที่สกัดได้ (รูปแบบ Pydantic/JSON)")
            print("="*50)
            print(profile_data.model_dump_json(indent=4))
            print("="*50)
        else:
            print("❌ ไม่สามารถสกัดข้อมูลได้")
    profile_data = dict(profile_data)


🚀 เริ่มกระบวนการสกัดข้อความจาก: ./input/kitichai - kitichai paichayon.pdf
📄 Page 1: อ่านด้วย Fitz ปกติสำเร็จ

🧠 ส่งข้อความ (ความยาว 836 ตัวอักษร) ให้ DSPy Agent วิเคราะห์เป็น JSON...


2026/03/20 05:03:33 WARNING dspy.primitives.module: Failed to set LM usage. Please return `dspy.Prediction` object from dspy.Module to enable usage tracking.



🎯 ผลลัพธ์ที่สกัดได้ (รูปแบบ Pydantic/JSON)
{
    "full_name": "Pokkitichai",
    "student_id": null,
    "email": "pokkitichai@gmail.com",
    "phone": "062-3875176",
    "location": "Pathum Thani, Thailand",
    "gpa": 3.02,
    "skill": [
        "ASP.Net",
        "HTML",
        "Python",
        "VueJS",
        "Javascript",
        "Sqlite",
        "PHP",
        "MsSQL",
        "Postman",
        "DBeaver",
        "Gitlab",
        "React",
        "NodeJS",
        "Docker",
        "VSCode",
        "Wordpress",
        "MySQL"
    ],
    "interests": [
        "IoT projects"
    ],
    "experience_summary": [
        "Nov 2022 - Present: Programmer at Jaroonrat Product, developing web applications using PHP, HTML, JavaScript, ASP.NET, NodeJS, React, VueJS, Python, and managing databases such as MsSQL, MySQL, and SQLite, while utilizing WordPress and tools like VSCode, Postman, Gitlab, Docker, and DBeaver.",
        "Aug 2018 - Oct 2022: Programmer at Sangthong Animal Fee

# 2. **Job Matching**

In [11]:
# pip install pandas numpy scikit-learn llama-index-embeddings-openai-like

In [2]:
import pandas as pd
import numpy as np
import httpx
import os
import json 
from sklearn.metrics.pairwise import cosine_similarity
from llama_index.embeddings.openai_like import OpenAILikeEmbedding

# ==========================================
# 1. Configuration & Setup BGE-M3

LLM_URL = 'https://llm.services.storemesh.com/v1'
API_KEY = 'sk-mtZ_gxsMzSoUndRivO6tew'

custom_http_client = httpx.Client(verify=False)
embedding_model = OpenAILikeEmbedding(
    model_name="bge-m3:latest",
    api_base=LLM_URL,
    api_key=API_KEY,
    http_client=custom_http_client 
)

# ==========================================
# 2. Main Job Matching Function (All-in-One)

def evaluate_candidate_job_match(
    candidate_skills: list, 
    master_csv_path: str = "match_job_course_v4 - skill_taxonomy.csv", 
    corpus_parquet_path: str = "./output/corpus.parquet",
    top_n: int = 3, 
    threshold: float = 0.75
):

    if not candidate_skills:
        return None, [{"status": "❌ ไม่พบทักษะของผู้สมัคร", "raw": "-", "mapped": "-", "score": 0}]
        
    if not os.path.exists(master_csv_path) or not os.path.exists(corpus_parquet_path):
        raise FileNotFoundError(f"ไม่พบไฟล์ Database (ต้องการ: {master_csv_path} และ {corpus_parquet_path})")

    raw_df = pd.read_csv(master_csv_path)
    corpus = pd.read_parquet(corpus_parquet_path)
    corpus['embedding'] = corpus['embedding'].apply(json.loads)
    master_vectors = np.array(corpus['embedding'].tolist())
    master_skill_names = corpus['skill_name'].tolist()
    matched_master_skills = set()
    candidate_match_details = []

    for c_skill in candidate_skills:
        c_vec = embedding_model.get_text_embedding(c_skill)
        c_vec_np = np.array(c_vec).reshape(1, -1)
        similarities = cosine_similarity(c_vec_np, master_vectors)[0]
        
        best_match_idx = np.argmax(similarities)
        best_score = similarities[best_match_idx]
        best_match_skill = master_skill_names[best_match_idx]
        
        if best_score >= threshold:
            matched_master_skills.add(best_match_skill)
            candidate_match_details.append({"raw": c_skill, "mapped": best_match_skill, "score": best_score, "status": "✅ ผ่าน"})
        else:
            candidate_match_details.append({"raw": c_skill, "mapped": best_match_skill, "score": best_score, "status": "❌ ปัดตก (คะแนนต่ำไป)"})

    if not matched_master_skills:
        return None, candidate_match_details

    job_scores = []
    jobs = raw_df['job_classification'].unique()

    for job in jobs:
        job_required_skills = set(raw_df[raw_df['job_classification'] == job]['skill_name'])
        skills_met = matched_master_skills.intersection(job_required_skills)
        
        skills_missing = list(job_required_skills - skills_met)
        MAX_MISSING_DISPLAY = 5
        if len(skills_missing) > MAX_MISSING_DISPLAY:
            missing_text = ", ".join(skills_missing[:MAX_MISSING_DISPLAY]) + f" ... (และอื่นๆ อีก {len(skills_missing) - MAX_MISSING_DISPLAY} ทักษะ)"
        else:
            missing_text = ", ".join(skills_missing) if skills_missing else "-"
        
        if len(job_required_skills) > 0:
            match_percent = (len(skills_met) / len(job_required_skills)) * 100
        else:
            match_percent = 0
            
        job_scores.append({
            'job_name': job,
            'skills_met_count': len(skills_met),
            'match_score': match_percent,
            'total_required': len(job_required_skills),
            'matched_skills': ", ".join(skills_met) if skills_met else "-",
            'missing_skills': missing_text 
        })

    results_df = pd.DataFrame(job_scores)
    top_matched_jobs = results_df.sort_values(
        by=['skills_met_count', 'match_score'], 
        ascending=[False, False]
    ).head(top_n)
    
    return top_matched_jobs, candidate_match_details


# ==========================================
# 3. การเรียกใช้งาน (ตัวอย่าง)
# ==========================================
if __name__ == "__main__":
    
    # สมมติข้อมูลที่ได้จาก Resume Parsing Agent
    candidate_extracted_skills = [
        "Hardware Troubleshooting",
        "Customer Service",
        "React",
        "NodeJS",
        "Sleeping" # ลองใส่คำแปลกๆ ดูว่า AI ปัดตกไหม
    ]

    # เรียกใช้แค่บรรทัดเดียว!
    results, mapping_details = evaluate_candidate_job_match(
        candidate_skills=candidate_extracted_skills,
        top_n=3,
        threshold=0.75
    )

    # --- ส่วน Print สรุปผล ---
    print("\n" + "="*60)
    print("🧠 [AI AGENT REPORT] รายงานการวิเคราะห์เรซูเม่และจับคู่งาน")
    print("="*60)

    print("\n🔍 1. การตีความทักษะ (Skill Semantic Mapping)")
    print("-" * 40)
    for detail in mapping_details:
        if detail['status'] == "✅ ผ่าน":
            print(f"  {detail['status']} '{detail['raw']}' -> ตีความเป็น '{detail['mapped']}' (ความมั่นใจ: {detail['score']:.2f})")
        else:
            print(f"  {detail['status']} '{detail['raw']}' -> คล้ายกับ '{detail['mapped']}' แต่ความมั่นใจแค่ {detail['score']:.2f}")

    print("\n🏆 2. ตำแหน่งงานที่เหมาะสมที่สุด (Top Job Recommendations)")
    print("-" * 40)
    if results is not None and not results.empty:
        for index, row in results.iterrows():
            print(f"\n💼 ตำแหน่ง: {row['job_name']}")
            print(f"   📊 ความเหมาะสม: {row['match_score']:.2f}% (ตรงสเปก {row['skills_met_count']}/{row['total_required']} ทักษะ)")
            print(f"   ✅ ทักษะที่มี: {row['matched_skills']}")
            print(f"   ❌ ทักษะที่ต้องพัฒนาเพิ่ม (Gap): {row['missing_skills']}")
    else:
        print("   ⚠️ ไม่พบตำแหน่งงานที่ผ่านเกณฑ์")
        
    print("\n" + "="*60)


🧠 [AI AGENT REPORT] รายงานการวิเคราะห์เรซูเม่และจับคู่งาน

🔍 1. การตีความทักษะ (Skill Semantic Mapping)
----------------------------------------
  ✅ ผ่าน 'Hardware Troubleshooting' -> ตีความเป็น 'Problem Solving' (ความมั่นใจ: 0.76)
  ✅ ผ่าน 'Customer Service' -> ตีความเป็น 'Customer Engagement' (ความมั่นใจ: 0.75)
  ✅ ผ่าน 'React' -> ตีความเป็น 'React Hooks' (ความมั่นใจ: 0.77)
  ✅ ผ่าน 'NodeJS' -> ตีความเป็น 'Node.js' (ความมั่นใจ: 0.91)
  ❌ ปัดตก (คะแนนต่ำไป) 'Sleeping' -> คล้ายกับ 'Spacing' แต่ความมั่นใจแค่ 0.65

🏆 2. ตำแหน่งงานที่เหมาะสมที่สุด (Top Job Recommendations)
----------------------------------------

💼 ตำแหน่ง: Software Developer and Prompt 
   📊 ความเหมาะสม: 1.14% (ตรงสเปก 2/175 ทักษะ)
   ✅ ทักษะที่มี: React Hooks, Node.js
   ❌ ทักษะที่ต้องพัฒนาเพิ่ม (Gap): Manage Scope, DevSecOps, Webhook API, Domain & Hosting, Test Planning ... (และอื่นๆ อีก 168 ทักษะ)

💼 ตำแหน่ง: Digital Transformation
   📊 ความเหมาะสม: 1.01% (ตรงสเปก 1/99 ทักษะ)
   ✅ ทักษะที่มี: Problem Solving
   ❌ ทั